# Lesson 15: How can we tell what AI made? — watermarks, deepfakes and ethics 🛡️
**Learning goals**
1. Understand why "spotting AI content" matters (deepfakes, cheating, fake news)
2. Build an **AI text watermark** yourself, and write a detector for it
3. Traffic-light scenarios: when is it OK to use AI?

**⏱️ Timing:** discussion 10' | watermark experiment 30' | scenarios + debate 20'

## 🖼️ An analogy first
Banknotes have **watermarks** you can only see against the light — they prevent forgery.
AI companies are researching "invisible watermarks" for AI-generated content:
- Images: hide a signal in the pixels that the human eye can't see (e.g. Google's SynthID)
- Text: make the AI **secretly prefer certain words**. It reads normally to humans, but a detector can measure it statistically

Today we'll add a text watermark to the rhyme writer from Lesson 5!

In [ ]:
import random, hashlib, re
from collections import defaultdict, Counter

corpus = """Twinkle, twinkle, little star, how I wonder what you are. Up above the world so high, like a diamond in the sky.
Mary had a little lamb, its fleece was white as snow. And everywhere that Mary went, the lamb was sure to go.
Humpty Dumpty sat on a wall. Humpty Dumpty had a great fall. All the king's horses and all the king's men could not put Humpty together again.
Jack and Jill went up the hill to fetch a pail of water. Jack fell down and broke his crown, and Jill came tumbling after.
Hey diddle diddle, the cat and the fiddle, the cow jumped over the moon. The little dog laughed to see such sport, and the dish ran away with the spoon.
Row, row, row your boat, gently down the stream. Merrily, merrily, merrily, merrily, life is but a dream.
Baa, baa, black sheep, have you any wool? Yes sir, yes sir, three bags full. One for the master, one for the dame, and one for the little boy who lives down the lane.
Little Bo Peep has lost her sheep and does not know where to find them. Leave them alone and they will come home, wagging their tails behind them.
Hickory dickory dock, the mouse ran up the clock. The clock struck one, the mouse ran down. Hickory dickory dock.
Rain, rain, go away, come again another day. Little Johnny wants to play.
Old Mother Hubbard went to the cupboard to fetch her poor dog a bone. When she got there the cupboard was bare, and so the poor dog had none.
Little Miss Muffet sat on a tuffet, eating her curds and whey. Along came a spider who sat down beside her and frightened Miss Muffet away.
Little Jack Horner sat in the corner, eating his Christmas pie. He put in his thumb and pulled out a plum and said, what a good boy am I.
The itsy bitsy spider went up the water spout. Down came the rain and washed the spider out. Out came the sun and dried up all the rain, and the itsy bitsy spider went up the spout again."""
import re
words = re.findall(r"[a-z']+|[.,?]", corpus.lower())

def join(ws):
    """Glue words back into text (no space before punctuation)."""
    return " ".join(ws).replace(" .", ".").replace(" ,", ",").replace(" ?", "?")

next_word = defaultdict(Counter)
for a, b in zip(words, words[1:]):
    next_word[a][b] += 1
print("The rhyme writer is ready")

## The watermark's secret rule 🟢
1. Using the **previous word** and a **secret key**, split all possible next words into "green" and "red" halves (a different split every time, like flipping coins)
2. When the watermarked AI generates text, it **prefers green words**
3. The detector uses the same key to count how many words in a text are green:
   - Ordinary text: about **50%** green (pure luck)
   - Watermarked text: **clearly more than 50%**

In [ ]:
SECRET = "my-secret-key"     # only the AI company knows this key

def is_green(prev, w):
    h = hashlib.sha256((SECRET + prev + "|" + w).encode()).digest()
    return h[0] % 2 == 0      # half of all words are green

def generate(start="the", length=60, watermark=False, boost=8):
    text = [start]
    while len(text) < length:
        options = next_word[text[-1]]
        if not options:
            text.append(random.choice(words)); continue   # dead end → jump to a random word
        ws = list(options)
        weights = [options[w] * (boost if watermark and is_green(text[-1], w) else 1) for w in ws]
        text.append(random.choices(ws, weights=weights)[0])
    return text

print("Normal:     ", join(generate("the", 30)))
print("Watermarked:", join(generate("the", 30, watermark=True)))

Can you tell the difference just by reading? Hardly at all! Now let's write the **detector**:

In [ ]:
def detect(ws):
    pairs = list(zip(ws, ws[1:]))
    green = sum(is_green(a, b) for a, b in pairs)
    ratio = green / len(pairs)
    z = (green - len(pairs) / 2) / (len(pairs) / 4) ** 0.5   # how many standard deviations above "lucky guessing"
    verdict = "🤖 probably watermarked" if z > 3 else "🧑 no watermark found"
    return f"green words {ratio:.0%}, z={z:.1f} → {verdict}"

print("Normal AI text:     ", detect(generate("the", 200)))
print("Watermarked AI text:", detect(generate("the", 200, watermark=True)))
print("Human nursery rhymes:", detect(words))

### 🧩 Exercise 1: Attack the watermark
Suppose someone tries to "wash out" the watermark:
1. **Change some of the words** in the watermarked text (e.g. replace every third word). Can the detector still find it?
2. Is the detector still accurate when the text is very short (10 words)? Why?
3. If someone doesn't know `SECRET`, can they fake a watermark?

In [ ]:
wm = generate("the", 200, watermark=True)
# replace every third word with a random word
attacked = [random.choice(words) if i % 3 == 0 else w for i, w in enumerate(wm)]
print("Original watermark:", detect(wm))
print("After editing:     ", detect(attacked))
print("Only 10 words:     ", detect(generate("the", 10, watermark=True)))

> 💡 Conclusion: watermarks are **useful but not perfect**. Rewriting and short texts both defeat them. That's also why "AI detectors" that accuse you of cheating **aren't always right** — and, the other way round, you can't count on changing a few words to "fool" your teacher. **Honesty** is the best strategy.

### 🚀 Challenge
Adjust `boost` (how strongly green words are preferred). What happens when it's too big (text quality)? Too small (detection)? Find a good balance.

## Part 2: Traffic-light scenarios 🚦
Run the cell below. For each scenario type **green** (fine), **amber** (be careful / it depends) or **red** (shouldn't do it), and explain why. There are no fixed answers — the **discussion** is what matters!

In [ ]:
scenarios = [
    "Using AI to explain the method for a maths problem I don't understand",
    "Getting AI to write my whole essay and handing it straight in",
    "Using AI to turn a classmate's photo into a funny meme and posting it in the group chat",
    "Using AI to check the grammar in my English essay",
    "Using AI to make up a quote 'said' by a celebrity and posting it online",
    "Using AI to make a poster and labelling it 'made with AI'",
    "Sending my home address and photo to an AI chatbot",
]
icons = {"green": "🟢", "amber": "🟡", "red": "🔴"}
answers = {}
for s in scenarios:
    answers[s] = input(f"{s}\n  → green / amber / red? ").strip().lower()
print("\nYour answers:")
for s, a in answers.items():
    print(icons.get(a, "⚪"), s)

## 💬 Class debate (pick one)
1. Is using AI for homework cheating?
2. Should AI-generated art be protected by copyright? If AI learned to paint by studying lots of artists' work, should those artists be paid?
3. Should the law require all AI-generated content to carry a watermark?

**Deepfake key points:** using AI to fake a **real person's** face or voice without their consent can hurt people, and is illegal in many countries.
If you see a suspicious video or voice message: **pause → check the source → don't share → tell a trusted adult**.

## 📌 Summary
- Watermark = make the AI secretly prefer "secretly chosen" words, then detect it with statistics
- Detection is **not foolproof**; honesty and labelling AI use are what really matter
- Respect others: don't fake real people, don't leak private information, label AI-generated content